<a href="https://colab.research.google.com/github/Deepakmewadaa/Anomaly-detection-in-Surveillance-Videos/blob/main/Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pedestrian Detection and Anomaly Detection Pipeline
## MOT17 Training | YOLOv5 Detector | DeepSORT Tracker | Avenue Dataset Inference

---

### Pipeline Overview
```
Train on MOT17 (normal pedestrian patterns)
         |
    YOLOv5 Detection
    (best of YOLOv5s / YOLOv5m / YOLOv5m-Frozen)
         |
  DeepSORT Multi-Object Tracking
  (Kalman Filter + Re-ID appearance features)
         |
  Rule-Based Anomaly Logic
  (Speed | Direction Change | Forbidden Zones)
         |
  Anomaly Flagging & Visualization
```


---
## Section 1: Mount Google Drive and Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully')


In [ ]:
import os, shutil, glob, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import configparser
import cv2
import yaml
from collections import defaultdict
from tqdm.notebook import tqdm

print('All base libraries imported')


In [ ]:
# ── Configure paths ─────────────────────────────────────────────────────────
# MOT17 dataset root on your Drive (adjust if needed)
DRIVE_MOT17_ROOT   = '/content/drive/MyDrive/Datasets/MOT17'

# Where the processed YOLO dataset will be saved
DRIVE_DATASET_ROOT = '/content/drive/MyDrive/Datasets/MOT17_YOLO'

# Local (Colab /content) working copy — faster I/O during training
LOCAL_DATASET_ROOT = '/content/MOT17_YOLO'

# Avenue dataset root on your Drive (for inference/tracking/anomaly)
DRIVE_AVENUE_ROOT  = '/content/drive/MyDrive/Datasets/Avenue'

# ── Output directory on Drive (weights, charts, video all saved here) ────────
OUT_DRIVE          = '/content/drive/MyDrive/MOT17_Avenue_Results'
os.makedirs(OUT_DRIVE, exist_ok=True)

# Detector variant to pick from MOT17 sequences (avoids duplicates)
CHOSEN_DETECTOR    = 'FRCNN'

# Train / Val split ratio
VAL_SPLIT          = 0.15

print(f'MOT17 source  : {DRIVE_MOT17_ROOT}')
print(f'YOLO dataset  : {DRIVE_DATASET_ROOT}')
print(f'Local copy    : {LOCAL_DATASET_ROOT}')
print(f'Avenue root   : {DRIVE_AVENUE_ROOT}')
print(f'Output dir    : {OUT_DRIVE}')
print(f'Detector      : {CHOSEN_DETECTOR}')


---
## Section 2: Explore the MOT17 Dataset


In [ ]:
# ── List available sequences ────────────────────────────────────────────────
mot17_train = os.path.join(DRIVE_MOT17_ROOT, 'train')
mot17_test  = os.path.join(DRIVE_MOT17_ROOT, 'test')

all_train_seqs = sorted(os.listdir(mot17_train)) if os.path.exists(mot17_train) else []
all_test_seqs  = sorted(os.listdir(mot17_test))  if os.path.exists(mot17_test)  else []

# Filter to chosen detector only
train_seqs = [s for s in all_train_seqs if CHOSEN_DETECTOR in s]
test_seqs  = [s for s in all_test_seqs  if CHOSEN_DETECTOR in s]

print(f'Train sequences ({len(train_seqs)}):')
for s in train_seqs: print(f'  {s}')
print(f'\nTest sequences ({len(test_seqs)}):')
for s in test_seqs:  print(f'  {s}')


In [ ]:
# ── Sequence summary ────────────────────────────────────────────────────────
def parse_seqinfo(seq_path):
    ini_path = os.path.join(seq_path, 'seqinfo.ini')
    cfg = configparser.ConfigParser()
    cfg.read(ini_path)
    s = cfg['Sequence']
    return {
        'name'  : s.get('name', '?'),
        'fps'   : s.get('frameRate', '?'),
        'frames': s.get('seqLength', '?'),
        'width' : s.get('imWidth', '?'),
        'height': s.get('imHeight', '?'),
    }

rows = []
for seq in train_seqs:
    info = parse_seqinfo(os.path.join(mot17_train, seq))
    gt_file = os.path.join(mot17_train, seq, 'gt', 'gt.txt')
    n_ann = 0
    if os.path.exists(gt_file):
        df = pd.read_csv(gt_file, header=None)
        df.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'cls', 'vis']
        n_ann = len(df[(df['cls'] == 1) & (df['conf'] == 1)])
    info['annotations'] = n_ann
    rows.append(info)

summary_df = pd.DataFrame(rows)
print('Dataset Summary (Train Sequences)\n')
print(summary_df.to_string(index=False))
print(f'\nTotal pedestrian annotations: {summary_df["annotations"].sum():,}')


In [ ]:
# ── Visualize a sample frame with ground-truth boxes ─────────────────────────
def visualize_gt_frame(seq_path, frame_idx=1):
    gt_file = os.path.join(seq_path, 'gt', 'gt.txt')
    img_dir = os.path.join(seq_path, 'img1')

    gt = pd.read_csv(gt_file, header=None,
                     names=['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'cls', 'vis'])
    gt = gt[(gt['cls'] == 1) & (gt['conf'] == 1) & (gt['frame'] == frame_idx)]

    img_file = os.path.join(img_dir, f'{frame_idx:06d}.jpg')
    img = Image.open(img_file)

    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.imshow(img)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(gt)))
    for (_, row), col in zip(gt.iterrows(), colors):
        rect = patches.Rectangle(
            (row['x'], row['y']), row['w'], row['h'],
            linewidth=2, edgecolor=col, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(row['x'], row['y'] - 4, f'ID:{int(row["id"])}',
                color=col, fontsize=7, fontweight='bold')
    ax.set_title(
        f'{os.path.basename(seq_path)}  |  Frame {frame_idx}  |  {len(gt)} pedestrians',
        fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if train_seqs:
    visualize_gt_frame(os.path.join(mot17_train, train_seqs[0]), frame_idx=1)


---
## Section 3: Data Preprocessing — MOT17 to YOLO Format
> Run on CPU runtime

### Format conversion
```
MOT17  :  frame, id, x_topleft, y_topleft, w, h, conf, cls, vis
YOLO   :  class_id  cx_norm  cy_norm  w_norm  h_norm
```


In [ ]:
# ── Conversion utility ──────────────────────────────────────────────────────
def mot_to_yolo(x, y, w, h, img_w, img_h):
    """Convert MOT top-left bbox to YOLO normalised center format."""
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    wn =  w           / img_w
    hn =  h           / img_h
    cx, cy, wn, hn = [max(0.0, min(1.0, v)) for v in (cx, cy, wn, hn)]
    return cx, cy, wn, hn


def process_sequence(seq_path, out_img_dir, out_lbl_dir,
                     frame_step=1, min_vis=0.3):
    """
    Convert one MOT17 sequence to YOLO-format images + labels.
    frame_step : sample every N-th frame (1 = all frames)
    min_vis    : skip annotations with visibility below this threshold
    """
    gt_file  = os.path.join(seq_path, 'gt', 'gt.txt')
    img_dir  = os.path.join(seq_path, 'img1')
    ini      = parse_seqinfo(seq_path)

    img_w, img_h = int(ini['width']), int(ini['height'])
    seq_name     = os.path.basename(seq_path)

    gt = pd.read_csv(gt_file, header=None,
                     names=['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'cls', 'vis'])
    gt = gt[(gt['cls'] == 1) & (gt['conf'] == 1) & (gt['vis'] >= min_vis)]

    frames = sorted(gt['frame'].unique())[::frame_step]

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    n_written = 0
    for fid in frames:
        src_img = os.path.join(img_dir, f'{fid:06d}.jpg')
        if not os.path.exists(src_img):
            continue

        dst_img = os.path.join(out_img_dir, f'{seq_name}_{fid:06d}.jpg')
        dst_lbl = os.path.join(out_lbl_dir, f'{seq_name}_{fid:06d}.txt')

        shutil.copy2(src_img, dst_img)

        rows_f = gt[gt['frame'] == fid]
        with open(dst_lbl, 'w') as f:
            for _, row in rows_f.iterrows():
                cx, cy, wn, hn = mot_to_yolo(
                    row['x'], row['y'], row['w'], row['h'], img_w, img_h
                )
                f.write(f'0 {cx:.6f} {cy:.6f} {wn:.6f} {hn:.6f}\n')
        n_written += 1

    return n_written


print('Conversion utilities defined')


In [ ]:
# ── Build folder structure ──────────────────────────────────────────────────
# frame_step=3 keeps dataset manageable; set to 1 to use every frame
FRAME_STEP = 3

split_dirs = {
    'train': {
        'images': os.path.join(DRIVE_DATASET_ROOT, 'train', 'images'),
        'labels': os.path.join(DRIVE_DATASET_ROOT, 'train', 'labels'),
    },
    'val': {
        'images': os.path.join(DRIVE_DATASET_ROOT, 'val', 'images'),
        'labels': os.path.join(DRIVE_DATASET_ROOT, 'val', 'labels'),
    },
    'test': {
        'images': os.path.join(DRIVE_DATASET_ROOT, 'test', 'images'),
    }
}
for split, paths in split_dirs.items():
    for p in paths.values():
        os.makedirs(p, exist_ok=True)

print('Directory structure created')
for split, paths in split_dirs.items():
    print(f'  {split}: {list(paths.values())}')


In [ ]:
# ── Process TRAIN sequences → temporary folder, then split ─────────────────
TEMP_DIR = '/content/mot17_temp'
TEMP_IMG = os.path.join(TEMP_DIR, 'images')
TEMP_LBL = os.path.join(TEMP_DIR, 'labels')
os.makedirs(TEMP_IMG, exist_ok=True)
os.makedirs(TEMP_LBL, exist_ok=True)

print(f'Processing {len(train_seqs)} train sequences (every {FRAME_STEP}th frame)\n')
total_frames = 0
for seq in tqdm(train_seqs, desc='Sequences'):
    seq_path = os.path.join(mot17_train, seq)
    n = process_sequence(seq_path, TEMP_IMG, TEMP_LBL,
                         frame_step=FRAME_STEP, min_vis=0.3)
    total_frames += n
    print(f'  {seq}: {n} frames written')

print(f'\nTotal frames processed: {total_frames:,}')


In [ ]:
# ── Train / Val split ───────────────────────────────────────────────────────
all_imgs = sorted(glob.glob(os.path.join(TEMP_IMG, '*.jpg')))
random.seed(42)
random.shuffle(all_imgs)

n_val    = int(len(all_imgs) * VAL_SPLIT)
val_imgs = all_imgs[:n_val]
trn_imgs = all_imgs[n_val:]


def move_split(img_list, dst_img_dir, dst_lbl_dir=None):
    for img_path in tqdm(img_list, leave=False):
        stem = Path(img_path).stem
        shutil.copy2(img_path, os.path.join(dst_img_dir, os.path.basename(img_path)))
        if dst_lbl_dir:
            lbl_path = os.path.join(TEMP_LBL, stem + '.txt')
            if os.path.exists(lbl_path):
                shutil.copy2(lbl_path, os.path.join(dst_lbl_dir, stem + '.txt'))


print(f'Train: {len(trn_imgs)} images | Val: {len(val_imgs)} images')
print('Copying train split ...')
move_split(trn_imgs, split_dirs['train']['images'], split_dirs['train']['labels'])
print('Copying val split ...')
move_split(val_imgs, split_dirs['val']['images'],   split_dirs['val']['labels'])
print('Train/Val split complete')


In [ ]:
# ── Process TEST sequences (images only — no GT labels in MOT17 test) ───────
print(f'Processing {len(test_seqs)} test sequences ...')
for seq in tqdm(test_seqs, desc='Test sequences'):
    seq_path = os.path.join(mot17_test, seq)
    img_dir  = os.path.join(seq_path, 'img1')
    imgs = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))[::FRAME_STEP]
    for src in imgs:
        dst = os.path.join(split_dirs['test']['images'],
                           f'{seq}_{os.path.basename(src)}')
        shutil.copy2(src, dst)

n_test = len(os.listdir(split_dirs['test']['images']))
print(f'Test images copied: {n_test:,}')


In [ ]:
# ── Create data.yaml ────────────────────────────────────────────────────────
yaml_path = os.path.join(DRIVE_DATASET_ROOT, 'data.yaml')
data_yaml = {
    'path' : DRIVE_DATASET_ROOT,
    'train': 'train/images',
    'val'  : 'val/images',
    'test' : 'test/images',
    'nc'   : 1,
    'names': ['pedestrian']
}
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'data.yaml written to: {yaml_path}')
print()
with open(yaml_path) as f:
    print(f.read())


In [ ]:
# ── Dataset statistics and sample visualization ──────────────────────────────
def count_split(img_dir, lbl_dir=None):
    n_imgs  = len(glob.glob(os.path.join(img_dir, '*.jpg')))
    n_boxes = 0
    if lbl_dir:
        for lf in glob.glob(os.path.join(lbl_dir, '*.txt')):
            with open(lf) as f:
                n_boxes += sum(1 for l in f if l.strip())
    return n_imgs, n_boxes

trn_n, trn_b = count_split(split_dirs['train']['images'], split_dirs['train']['labels'])
val_n, val_b = count_split(split_dirs['val']['images'],   split_dirs['val']['labels'])
tst_n, _     = count_split(split_dirs['test']['images'])

print('Final Dataset Statistics')
print('-' * 40)
print(f'  Train : {trn_n:5,} images | {trn_b:6,} annotations')
print(f'  Val   : {val_n:5,} images | {val_b:6,} annotations')
print(f'  Test  : {tst_n:5,} images | (no GT)')
print('-' * 40)

def show_yolo_sample(img_dir, lbl_dir, n=4):
    imgs = glob.glob(os.path.join(img_dir, '*.jpg'))
    chosen = random.sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(1, len(chosen), figsize=(5 * len(chosen), 5))
    if len(chosen) == 1:
        axes = [axes]
    for ax, ip in zip(axes, chosen):
        img = np.array(Image.open(ip))
        h, w = img.shape[:2]
        lp = os.path.join(lbl_dir, Path(ip).stem + '.txt')
        ax.imshow(img)
        if os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    _, cx, cy, bw, bh = map(float, line.split())
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    rect = patches.Rectangle(
                        (x1, y1), bw * w, bh * h,
                        linewidth=2, edgecolor='lime', facecolor='none')
                    ax.add_patch(rect)
        ax.set_title(Path(ip).stem[-12:], fontsize=8)
        ax.axis('off')
    plt.suptitle('YOLO-formatted samples (green = pedestrian bbox)', fontsize=12)
    plt.tight_layout()
    plt.show()

show_yolo_sample(split_dirs['train']['images'], split_dirs['train']['labels'])


---
## Section 4: YOLOv5 Training

Three variants are trained for comparison:
1. **YOLOv5s** — lightweight baseline
2. **YOLOv5m** — balanced accuracy and speed
3. **YOLOv5m (frozen backbone)** — transfer learning with backbone layers 0-9 frozen

The best-performing model (by mAP@0.5) will be used for Avenue inference.


In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — switch runtime to T4 GPU before training')


In [ ]:
!git clone https://github.com/ultralytics/yolov5.git /content/yolov5 --depth 1 -q
!pip install -q -r /content/yolov5/requirements.txt
print('YOLOv5 ready')


In [ ]:
# ── Copy dataset locally for faster I/O during training ─────────────────────
if not os.path.exists(LOCAL_DATASET_ROOT):
    print('Copying dataset from Drive to local storage ...')
    shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
    local_yaml = os.path.join(LOCAL_DATASET_ROOT, 'data.yaml')
    data_yaml['path'] = LOCAL_DATASET_ROOT
    with open(local_yaml, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    print('Dataset copied locally')
else:
    print('Local dataset already exists')
    local_yaml = os.path.join(LOCAL_DATASET_ROOT, 'data.yaml')

print(f'data.yaml path : {local_yaml}')


In [ ]:
# ── Training configuration ──────────────────────────────────────────────────
EPOCHS     = 30
IMG_SIZE   = 640
BATCH_SIZE = 16    # reduce to 8 if out of memory
WORKERS    = 4

# Training runs saved directly to Google Drive
RESULTS_DIR = os.path.join(OUT_DRIVE, 'training_runs')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Epochs     : {EPOCHS}')
print(f'Image size : {IMG_SIZE}')
print(f'Batch size : {BATCH_SIZE}')


In [ ]:
# ── Model 1: YOLOv5s (small) ────────────────────────────────────────────────
print('Training YOLOv5s ...')
!python /content/yolov5/train.py \
    --img {IMG_SIZE} \
    --batch {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --data {local_yaml} \
    --weights yolov5s.pt \
    --name yolov5s_mot17 \
    --workers {WORKERS} \
    --project {RESULTS_DIR} \
    --exist-ok \
    --cache ram


In [ ]:
# ── Model 2: YOLOv5m (medium) ───────────────────────────────────────────────
print('Training YOLOv5m ...')
!python /content/yolov5/train.py \
    --img {IMG_SIZE} \
    --batch {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --data {local_yaml} \
    --weights yolov5m.pt \
    --name yolov5m_mot17 \
    --workers {WORKERS} \
    --project {RESULTS_DIR} \
    --exist-ok \
    --cache ram


In [ ]:
# ── Model 3: YOLOv5m with frozen backbone (layers 0-9) ──────────────────────
print('Training YOLOv5m (frozen backbone, layers 0-9) ...')
!python /content/yolov5/train.py \
    --img {IMG_SIZE} \
    --batch {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --data {local_yaml} \
    --weights yolov5m.pt \
    --name yolov5m_frozen_mot17 \
    --freeze 10 \
    --workers {WORKERS} \
    --project {RESULTS_DIR} \
    --exist-ok \
    --cache ram


---
## Section 5: Compare Training Results
> Training curves and best-epoch metrics for all three YOLOv5 variants


In [ ]:
# ── Load results CSVs ───────────────────────────────────────────────────────
model_configs = {
    'YOLOv5s'        : os.path.join(RESULTS_DIR, 'yolov5s_mot17',        'results.csv'),
    'YOLOv5m'        : os.path.join(RESULTS_DIR, 'yolov5m_mot17',        'results.csv'),
    'YOLOv5m-Frozen' : os.path.join(RESULTS_DIR, 'yolov5m_frozen_mot17', 'results.csv'),
}

results = {}
for name, csv_path in model_configs.items():
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        results[name] = df
        print(f'{name}: {len(df)} epochs loaded')
    else:
        print(f'{name}: results.csv not found at {csv_path}')


In [ ]:
# ── Training curves comparison ──────────────────────────────────────────────
COLORS = {'YOLOv5s': '#2196F3', 'YOLOv5m': '#4CAF50', 'YOLOv5m-Frozen': '#FF9800'}

COL_MAP = {
    'mAP50'    : ['metrics/mAP_0.5',     'metrics/mAP50'],
    'mAP5095'  : ['metrics/mAP_0.5:0.95','metrics/mAP50-95'],
    'Precision': ['metrics/precision',    'Precision'],
    'Recall'   : ['metrics/recall',       'Recall'],
    'BoxLoss'  : ['train/box_loss',       'train/box_om'],
    'ObjLoss'  : ['train/obj_loss',       'train/obj_om'],
}

def get_col(df, aliases):
    for a in aliases:
        if a in df.columns:
            return df[a]
    return None

metrics_to_plot = ['mAP50', 'mAP5095', 'Precision', 'Recall', 'BoxLoss', 'ObjLoss']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, metric in zip(axes, metrics_to_plot):
    for name, df in results.items():
        series = get_col(df, COL_MAP[metric])
        if series is not None:
            ax.plot(series.values, label=name, color=COLORS[name], linewidth=2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('YOLOv5 Model Comparison — MOT17 Pedestrian Detection',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DRIVE, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Comparison chart saved to: {os.path.join(OUT_DRIVE, "model_comparison.png")}')


In [ ]:
# ── Summary table — best epoch metrics ─────────────────────────────────────
summary_rows = []
for name, df in results.items():
    row = {'Model': name}
    for metric, aliases in COL_MAP.items():
        s = get_col(df, aliases)
        row[metric] = f'{s.max():.4f}' if s is not None else 'N/A'
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('Model')
print('\nBest-epoch Metrics Comparison\n')
print(summary.to_string())


In [ ]:
# ── Select best model by mAP@0.5 ────────────────────────────────────────────
best_model_name = None
best_map50      = -1.0

for name, df in results.items():
    s = get_col(df, COL_MAP['mAP50'])
    if s is not None and s.max() > best_map50:
        best_map50      = s.max()
        best_model_name = name

name_to_folder = {
    'YOLOv5s'        : 'yolov5s_mot17',
    'YOLOv5m'        : 'yolov5m_mot17',
    'YOLOv5m-Frozen' : 'yolov5m_frozen_mot17',
}

BEST_WEIGHTS = os.path.join(
    RESULTS_DIR,
    name_to_folder.get(best_model_name, 'yolov5m_mot17'),
    'weights', 'best.pt'
)

print(f'Best model     : {best_model_name}')
print(f'Best mAP@0.5   : {best_map50:.4f}')
print(f'Weights path   : {BEST_WEIGHTS}')


---
## Section 6: Inference on Avenue Dataset
> The best YOLOv5 model from training is applied to the CUHK Avenue dataset for
> pedestrian detection, DeepSORT tracking, and anomaly analysis.

The Avenue dataset consists of surveillance camera footage of a building avenue
with normal and abnormal pedestrian behaviour (running, throwing bags, unusual
walking patterns).


In [ ]:
# ── Install DeepSORT ────────────────────────────────────────────────────────
import shutil, os

# Remove stale clone if it exists
if os.path.exists('/content/yolo_tracking'):
    shutil.rmtree('/content/yolo_tracking')

!git clone https://github.com/mikel-brostrom/yolo_tracking.git /content/yolo_tracking -q

# requirements.txt location changed in newer versions of the repo — check both
if os.path.exists('/content/yolo_tracking/requirements.txt'):
    !pip install -q -r /content/yolo_tracking/requirements.txt
elif os.path.exists('/content/yolo_tracking/boxmot/requirements.txt'):
    !pip install -q -r /content/yolo_tracking/boxmot/requirements.txt
else:
    # Fallback: install core deps directly
    !pip install -q deep-sort-realtime

!pip install -q gdown

print('DeepSORT dependencies installed')

In [ ]:
# ── Load YOLOv5 model for frame-by-frame inference ──────────────────────────
import sys
import torch
sys.path.insert(0, '/content/yolov5')

from models.common      import DetectMultiBackend
from utils.general      import non_max_suppression
from utils.torch_utils  import select_device
from utils.augmentations import letterbox

device = select_device('0' if torch.cuda.is_available() else 'cpu')
yolo_model = DetectMultiBackend(BEST_WEIGHTS, device=device)
yolo_model.eval()
print(f'YOLOv5 model loaded on {device}')
print(f'Using weights: {BEST_WEIGHTS}')


def detect_frame(img_bgr, conf_thres=0.4, iou_thres=0.45, img_size=640):
    """
    Run YOLOv5 on a single BGR frame.
    Returns list of [x1, y1, x2, y2, confidence] for each detection.
    """
    img_lb, ratio, pad = letterbox(img_bgr, img_size,
                                   stride=yolo_model.stride, auto=True)
    img_t = torch.from_numpy(img_lb.transpose(2, 0, 1)).float().to(device) / 255.0
    img_t = img_t.unsqueeze(0)

    with torch.no_grad():
        pred = yolo_model(img_t)
    det = non_max_suppression(pred, conf_thres, iou_thres)[0]

    detections = []
    if det is not None and len(det):
        for *xyxy, conf, cls in det.cpu().numpy():
            x1 = (xyxy[0] - pad[0]) / ratio[0]
            y1 = (xyxy[1] - pad[1]) / ratio[1]
            x2 = (xyxy[2] - pad[0]) / ratio[0]
            y2 = (xyxy[3] - pad[1]) / ratio[1]
            detections.append([float(x1), float(y1), float(x2), float(y2), float(conf)])
    return detections


In [ ]:
# ── DeepSORT Tracker Setup ───────────────────────────────────────────────────
# DeepSORT combines a Kalman Filter (motion model) with a deep appearance
# descriptor (Re-ID network) for robust identity-preserving tracking.
#
# Key parameters:
#   max_dist        — maximum cosine distance for appearance matching
#   min_confidence  — detections below this confidence are ignored
#   max_iou_distance — IoU gate for spatial matching
#   max_age         — frames a track can be lost before deletion
#   n_init          — consecutive detections to confirm a new track
#   nn_budget       — max samples per class stored in the appearance gallery

sys.path.insert(0, '/content/yolo_tracking')

from deep_sort_realtime.deepsort_tracker import DeepSort

deepsort = DeepSort(
    max_age=30,
    n_init=3,
    nms_max_overlap=0.7,
    max_cosine_distance=0.4,
    nn_budget=100,
    embedder='mobilenet',          # lightweight Re-ID backbone
    half=True,                     # fp16 on GPU
    bgr=True,                      # input frames are BGR (OpenCV default)
    embedder_gpu=torch.cuda.is_available(),
)

print('DeepSORT tracker initialised')
print('  Re-ID backbone  : MobileNet')
print(f'  Max track age   : 30 frames')
print(f'  Confirm after   : 3 consecutive detections')


In [ ]:
# ── Avenue Dataset Configuration ────────────────────────────────────────────
# The Avenue dataset is structured as:
#   Avenue/
#     testing_videos/   — .avi files (01.avi ... 21.avi)
#     training_videos/  — .avi files (01.avi ... 16.avi)
#
# We extract frames from a test video and run the full pipeline on it.
# Adjust AVENUE_VIDEO to point to whichever clip you want to analyse.

AVENUE_VIDEO_DIR = os.path.join(DRIVE_AVENUE_ROOT, 'testing_videos')
AVENUE_FRAMES_DIR = '/content/avenue_frames'

# List available test videos
if os.path.exists(AVENUE_VIDEO_DIR):
    avenue_videos = sorted(glob.glob(os.path.join(AVENUE_VIDEO_DIR, '*.avi')))
    print(f'Avenue test videos found: {len(avenue_videos)}')
    for v in avenue_videos:
        print(f'  {os.path.basename(v)}')
else:
    avenue_videos = []
    print(f'Avenue video directory not found: {AVENUE_VIDEO_DIR}')
    print('Update DRIVE_AVENUE_ROOT in Section 1 to point to your Avenue dataset.')

# Select the first video by default — change index to pick another
AVENUE_VIDEO = avenue_videos[0] if avenue_videos else None
print(f'\nSelected video : {AVENUE_VIDEO}')


In [ ]:
# ── Extract frames from the selected Avenue video ───────────────────────────
N_FRAMES = 200   # number of frames to process (set None for the full video)

os.makedirs(AVENUE_FRAMES_DIR, exist_ok=True)

if AVENUE_VIDEO is None:
    raise RuntimeError('No Avenue video found. Check DRIVE_AVENUE_ROOT path.')

cap = cv2.VideoCapture(AVENUE_VIDEO)
total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps          = cap.get(cv2.CAP_PROP_FPS)
video_w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

limit = N_FRAMES if N_FRAMES else total_video_frames
frame_paths = []

for idx in range(limit):
    ret, frame = cap.read()
    if not ret:
        break
    fp = os.path.join(AVENUE_FRAMES_DIR, f'{idx + 1:06d}.jpg')
    cv2.imwrite(fp, frame)
    frame_paths.append(fp)

cap.release()

print(f'Video           : {os.path.basename(AVENUE_VIDEO)}')
print(f'Total frames    : {total_video_frames}')
print(f'Frames extracted: {len(frame_paths)}')
print(f'Resolution      : {video_w} x {video_h}')
print(f'FPS             : {video_fps:.1f}')

all_frames = frame_paths
h_img, w_img = video_h, video_w


In [ ]:
# ── Run YOLOv5 Detection + DeepSORT Tracking ────────────────────────────────
# For each frame:
#   1. YOLOv5 detects pedestrian bounding boxes
#   2. DeepSORT associates detections to existing tracks using:
#      - Kalman Filter predicted positions (motion model)
#      - MobileNet appearance embeddings (Re-ID)
#   3. Track histories record each pedestrian's centre-point trajectory

track_history = defaultdict(list)   # track_id -> [(frame_id, (cx, cy)), ...]
all_results   = {}                  # frame_id -> [(track_id, [x1,y1,x2,y2]), ...]

print(f'Running YOLOv5 + DeepSORT on {len(all_frames)} frames ...')
print(f'Detector confidence threshold : 0.4')
print(f'DeepSORT max track age        : 30 frames')
print()

for fid, fp in enumerate(tqdm(all_frames, desc='Tracking'), start=1):
    img_bgr = cv2.imread(fp)
    if img_bgr is None:
        continue

    # Step 1: YOLOv5 detection — returns [[x1,y1,x2,y2,conf], ...]
    raw_dets = detect_frame(img_bgr)

    # Step 2: Format for DeepSORT — list of ([x1,y1,x2,y2], confidence, class_id)
    ds_input = [([d[0], d[1], d[2], d[3]], d[4], 0) for d in raw_dets]

    # Step 3: DeepSORT update — returns confirmed track objects
    tracks = deepsort.update_tracks(ds_input, frame=img_bgr)

    active_this_frame = []
    for track in tracks:
        if not track.is_confirmed():
            continue
        tid  = track.track_id
        bbox = track.to_ltrb()   # [x1, y1, x2, y2]
        active_this_frame.append((tid, bbox))

        cx = (bbox[0] + bbox[2]) / 2
        cy = (bbox[1] + bbox[3]) / 2
        track_history[tid].append((fid, (cx, cy)))

    all_results[fid] = active_this_frame

total_unique_ids = len(track_history)
print(f'Tracking complete')
print(f'Total unique pedestrian IDs : {total_unique_ids}')
print(f'Total frames processed      : {len(all_results)}')


In [ ]:
# ── Visualise DeepSORT tracking trajectories ─────────────────────────────────
LONG_TRACKS = {tid: hist for tid, hist in track_history.items() if len(hist) >= 10}

mid_fid = len(all_frames) // 2
mid_img = cv2.cvtColor(cv2.imread(all_frames[mid_fid]), cv2.COLOR_BGR2RGB)

cmap = plt.cm.rainbow(np.linspace(0, 1, len(LONG_TRACKS)))
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(mid_img)

for (tid, hist), col in zip(LONG_TRACKS.items(), cmap):
    pts = [p for _, p in hist]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, '-', color=col, linewidth=1.5, alpha=0.8)
    ax.plot(xs[-1], ys[-1], 'o', color=col, markersize=5)
    ax.text(xs[-1] + 3, ys[-1] - 3, str(tid), color=col,
            fontsize=7, fontweight='bold')

video_name = os.path.basename(AVENUE_VIDEO) if AVENUE_VIDEO else 'Avenue'
ax.set_title(f'DeepSORT Tracking Trajectories — {video_name}  '
             f'(frames 1-{len(all_frames)})', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DRIVE, 'tracking_trajectories_avenue.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Trajectory plot saved to: {os.path.join(OUT_DRIVE, "tracking_trajectories_avenue.png")}')


---
## Section 7: Rule-Based Anomaly Detection

Three complementary rules are applied to each DeepSORT track trajectory:

| Rule | Trigger condition |
|------|------------------|
| **Speed** | Instantaneous displacement > threshold px/frame |
| **Direction Change** | Heading reversal > threshold degrees between consecutive steps |
| **Forbidden Zone** | Track centroid enters a defined rectangular region |

These rules operate on tracked trajectories rather than raw detections, so
DeepSORT's stable identity assignment directly improves anomaly precision
(fewer ID switches mean fewer spurious anomaly triggers).


In [ ]:
# ── Anomaly detection utilities ─────────────────────────────────────────────

# Thresholds — tune these for the Avenue camera and scene geometry
SPEED_THRESH      = 40.0   # px/frame — above this flags a speed anomaly
DIR_CHANGE_THRESH = 120.0  # degrees  — sudden heading reversal threshold

# Forbidden zone defined as a fraction of image dimensions (x1, y1, x2, y2)
# Example: top-left 25% of the frame. Set to None to disable.
FORBIDDEN_ZONE_FRAC = (0.0, 0.0, 0.25, 0.25)


def compute_speed(p1, p2):
    return math.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)


def compute_angle(v1, v2):
    """Angle in degrees between two 2D displacement vectors."""
    dot = v1[0] * v2[0] + v1[1] * v2[1]
    mag = math.sqrt(v1[0] ** 2 + v1[1] ** 2) * math.sqrt(v2[0] ** 2 + v2[1] ** 2)
    if mag < 1e-6:
        return 0.0
    return math.degrees(math.acos(max(-1.0, min(1.0, dot / mag))))


def in_forbidden_zone(cx, cy, frac, img_w, img_h):
    if frac is None:
        return False
    x1, y1, x2, y2 = (frac[0] * img_w, frac[1] * img_h,
                       frac[2] * img_w, frac[3] * img_h)
    return x1 <= cx <= x2 and y1 <= cy <= y2


def analyze_track(tid, history, img_w, img_h,
                  speed_thresh=SPEED_THRESH,
                  dir_thresh=DIR_CHANGE_THRESH,
                  forbidden_frac=FORBIDDEN_ZONE_FRAC):
    """
    Analyse one track's trajectory for anomalies.

    Returns:
        anomaly_frames    : dict[frame_id] = list of anomaly type strings
        max_speed         : maximum instantaneous speed observed
        max_dir_change    : maximum direction change observed (degrees)
        zone_entry_frames : list of frame IDs where the track entered the zone
    """
    anomaly_frames = defaultdict(list)
    speeds         = []
    dir_changes    = []
    zone_frames    = []

    pts = [(fid, cx, cy) for fid, (cx, cy) in history]

    for i in range(1, len(pts)):
        fid, cx, cy = pts[i]
        _, px, py   = pts[i - 1]

        spd = compute_speed((px, py), (cx, cy))
        speeds.append(spd)
        if spd > speed_thresh:
            anomaly_frames[fid].append(f'SPEED({spd:.1f})')

        if i >= 2:
            _, ppx, ppy = pts[i - 2]
            v1  = (px - ppx, py - ppy)
            v2  = (cx - px,  cy - py)
            ang = compute_angle(v1, v2)
            dir_changes.append(ang)
            if ang > dir_thresh:
                anomaly_frames[fid].append(f'DIR({ang:.1f}deg)')

        if in_forbidden_zone(cx, cy, forbidden_frac, img_w, img_h):
            zone_frames.append(fid)
            anomaly_frames[fid].append('FORBIDDEN_ZONE')

    return {
        'anomaly_frames'    : dict(anomaly_frames),
        'max_speed'         : max(speeds)      if speeds      else 0.0,
        'max_dir_change'    : max(dir_changes) if dir_changes else 0.0,
        'zone_entry_frames' : zone_frames,
    }


print('Anomaly detection utilities defined')


In [ ]:
# ── Run anomaly analysis on all DeepSORT tracks ──────────────────────────────
anomaly_report = {}

for tid, hist in track_history.items():
    if len(hist) < 3:
        continue
    result = analyze_track(tid, hist, w_img, h_img)
    anomaly_report[tid] = result

all_speeds      = [r['max_speed']      for r in anomaly_report.values()]
all_dirs        = [r['max_dir_change'] for r in anomaly_report.values()]
total_anomalous = sum(1 for r in anomaly_report.values() if r['anomaly_frames'])

print('Anomaly Detection Summary')
print('-' * 50)
print(f'  Total tracks analysed     : {len(anomaly_report)}')
print(f'  Tracks with anomalies     : {total_anomalous}')
print(f'  Anomaly rate              : '
      f'{total_anomalous / max(1, len(anomaly_report)) * 100:.1f}%')
print(f'  Mean max speed (px/frame) : {np.mean(all_speeds):.2f}')
print(f'  Max speed observed        : {max(all_speeds):.2f}  (thresh={SPEED_THRESH})')
print(f'  Max direction change (deg): {max(all_dirs):.1f}    (thresh={DIR_CHANGE_THRESH})')

type_counts = defaultdict(int)
for r in anomaly_report.values():
    for events in r['anomaly_frames'].values():
        for e in events:
            kind = e.split('(')[0]
            type_counts[kind] += 1

print('\n  Anomaly events by type:')
for k, v in sorted(type_counts.items()):
    print(f'    {k:<20s}: {v}')


In [ ]:
# ── Visualise anomaly detections on a frame ──────────────────────────────────
def draw_anomaly_frame(frame_id, frame_path, results_dict, anomaly_report,
                       img_w, img_h, forbidden_frac):
    img = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(img)

    if forbidden_frac:
        x1, y1 = forbidden_frac[0] * img_w, forbidden_frac[1] * img_h
        x2, y2 = forbidden_frac[2] * img_w, forbidden_frac[3] * img_h
        zone_rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor='red', facecolor='red', alpha=0.15)
        ax.add_patch(zone_rect)
        ax.text(x1 + 5, y1 + 15, 'FORBIDDEN', color='red',
                fontsize=9, fontweight='bold')

    active = results_dict.get(frame_id, [])
    for tid, bbox in active:
        x1, y1, x2, y2 = bbox
        anomalies = (anomaly_report.get(tid, {})
                                   .get('anomaly_frames', {})
                                   .get(frame_id, []))
        color = 'red'  if anomalies else 'lime'
        lw    = 3      if anomalies else 1.5
        rect  = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=lw, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        label = f'ID:{tid}' + (f' [{anomalies[0]}]' if anomalies else '')
        ax.text(x1, y1 - 4, label, color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.4))

    ax.set_title(f'Frame {frame_id} — Red = Anomaly  |  Green = Normal', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


anomaly_fids = sorted(set(
    fid for r in anomaly_report.values() for fid in r['anomaly_frames']))

show_fid = anomaly_fids[0] if anomaly_fids else (len(all_frames) // 2)
show_fp  = all_frames[show_fid - 1] if show_fid <= len(all_frames) else all_frames[-1]

draw_anomaly_frame(show_fid, show_fp, all_results, anomaly_report,
                   w_img, h_img, FORBIDDEN_ZONE_FRAC)


In [ ]:
# ── Anomaly metric distributions ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_speeds, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(SPEED_THRESH, color='red', linestyle='--', linewidth=2,
                label=f'Threshold ({SPEED_THRESH} px/frame)')
axes[0].set_title('Max Speed per Track (px/frame)', fontsize=12)
axes[0].set_xlabel('Speed')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(all_dirs, bins=30, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(DIR_CHANGE_THRESH, color='red', linestyle='--', linewidth=2,
                label=f'Threshold ({DIR_CHANGE_THRESH} deg)')
axes[1].set_title('Max Direction Change per Track (degrees)', fontsize=12)
axes[1].set_xlabel('Degrees')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Anomaly Metric Distributions — Avenue Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DRIVE, 'anomaly_distributions_avenue.png'), dpi=150, bbox_inches='tight')
plt.show()


---
## Section 8: Export Annotated Video
> Renders the full tracked and anomaly-annotated sequence as an MP4


In [ ]:
import cv2

OUT_VIDEO = os.path.join(OUT_DRIVE, 'tracked_anomaly_avenue.mp4')
out_fps   = int(video_fps) if video_fps > 0 else 10

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
vw     = cv2.VideoWriter(OUT_VIDEO, fourcc, out_fps, (w_img, h_img))

TRACK_COLORS = {}   # tid -> BGR colour

for fid, fp in enumerate(tqdm(all_frames, desc='Writing video'), start=1):
    frame = cv2.imread(fp)
    if frame is None:
        continue

    # Draw forbidden zone
    if FORBIDDEN_ZONE_FRAC:
        x1z = int(FORBIDDEN_ZONE_FRAC[0] * w_img)
        y1z = int(FORBIDDEN_ZONE_FRAC[1] * h_img)
        x2z = int(FORBIDDEN_ZONE_FRAC[2] * w_img)
        y2z = int(FORBIDDEN_ZONE_FRAC[3] * h_img)
        overlay = frame.copy()
        cv2.rectangle(overlay, (x1z, y1z), (x2z, y2z), (0, 0, 200), -1)
        frame = cv2.addWeighted(overlay, 0.25, frame, 0.75, 0)
        cv2.rectangle(frame, (x1z, y1z), (x2z, y2z), (0, 0, 255), 2)
        cv2.putText(frame, 'FORBIDDEN', (x1z + 4, y1z + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

    active = all_results.get(fid, [])
    for tid, bbox in active:
        if tid not in TRACK_COLORS:
            rng = np.random.default_rng(int(str(tid), 36) % (2**31))
            TRACK_COLORS[tid] = tuple(rng.integers(50, 255, 3).tolist())

        x1, y1, x2, y2 = [int(v) for v in bbox]
        anoms = (anomaly_report.get(tid, {})
                               .get('anomaly_frames', {})
                               .get(fid, []))
        col = (0, 0, 255) if anoms else TRACK_COLORS[tid]
        lw  = 3 if anoms else 2

        cv2.rectangle(frame, (x1, y1), (x2, y2), col, lw)
        label = f'ID:{tid}' + (' [ANOMALY]' if anoms else '')
        cv2.putText(frame, label, (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1, cv2.LINE_AA)

        # Draw motion tail (last 20 positions)
        hist = track_history.get(tid, [])
        tail = [(int(cx), int(cy))
                for f, (cx, cy) in hist[-20:]
                if f <= fid]
        for j in range(1, len(tail)):
            cv2.line(frame, tail[j - 1], tail[j], TRACK_COLORS[tid], 1)

    cv2.putText(frame, f'Frame {fid}', (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    vw.write(frame)

vw.release()
print(f'Video saved to {OUT_VIDEO}')


---
## Section 9: Save All Results to Google Drive


In [ ]:
# ── Verify all outputs were saved to Drive ──────────────────────────────────
# All files are written directly to Google Drive throughout the notebook.
# This cell just confirms everything is in place.

print(f'Checking outputs in: {OUT_DRIVE}')
print()

expected_files = [
    'model_comparison.png',
    'tracking_trajectories_avenue.png',
    'anomaly_distributions_avenue.png',
    'final_comparison_bar.png',
    'tracked_anomaly_avenue.mp4',
]

for fname in expected_files:
    fpath = os.path.join(OUT_DRIVE, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  [OK]  {fname}  ({size_mb:.1f} MB)')
    else:
        print(f'  [--]  {fname}  (not yet generated — run earlier cells)')

# Verify trained weights
print()
weights_dir = os.path.join(OUT_DRIVE, 'training_runs')
for model_name in ['yolov5s_mot17', 'yolov5m_mot17', 'yolov5m_frozen_mot17']:
    best_w = os.path.join(weights_dir, model_name, 'weights', 'best.pt')
    last_w = os.path.join(weights_dir, model_name, 'weights', 'last.pt')
    if os.path.exists(best_w):
        size_mb = os.path.getsize(best_w) / 1e6
        print(f'  [OK]  {model_name}/weights/best.pt  ({size_mb:.1f} MB)')
    else:
        print(f'  [--]  {model_name}/weights/best.pt  (not yet trained)')
    if os.path.exists(last_w):
        size_mb = os.path.getsize(last_w) / 1e6
        print(f'  [OK]  {model_name}/weights/last.pt  ({size_mb:.1f} MB)')

print(f'\nAll outputs directory: {OUT_DRIVE}')


---
## Section 10: Final Summary — Model Comparison and Pipeline Review


In [ ]:
# ── Model parameter counts ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/yolov5')
from models.yolo import Model


def count_params(cfg, nc=1):
    try:
        m = Model(cfg, ch=3, nc=nc)
        total     = sum(p.numel() for p in m.parameters())
        trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
        return total, trainable
    except Exception:
        return 0, 0


param_data = [
    ('YOLOv5s',        '/content/yolov5/models/yolov5s.yaml', 'all layers trainable'),
    ('YOLOv5m',        '/content/yolov5/models/yolov5m.yaml', 'all layers trainable'),
    ('YOLOv5m-Frozen', '/content/yolov5/models/yolov5m.yaml', 'backbone layers 0-9 frozen'),
]

print(f'{"Model":<22} {"Total Params":>14}   Strategy')
print('-' * 62)
for name, cfg, strat in param_data:
    tot, _ = count_params(cfg)
    print(f'{name:<22} {tot:>14,}   {strat}')
print('-' * 62)
print('\nNote: YOLOv5m-Frozen has the same total parameter count as YOLOv5m')
print('but only neck and head weights were updated during training.')


In [ ]:
# ── Final comparison bar chart ──────────────────────────────────────────────
if results:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    models_list = list(results.keys())
    map50_vals  = []
    prec_vals   = []

    for name, df in results.items():
        ms = get_col(df, COL_MAP['mAP50'])
        ps = get_col(df, COL_MAP['Precision'])
        map50_vals.append(ms.max() if ms is not None else 0)
        prec_vals.append(ps.max()  if ps is not None else 0)

    bar_colors = ['#2196F3', '#4CAF50', '#FF9800']
    x = np.arange(len(models_list))

    axes[0].bar(x, map50_vals, color=bar_colors, edgecolor='white', linewidth=1.2)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(models_list, rotation=10)
    axes[0].set_ylabel('mAP@0.5')
    axes[0].set_title('Best mAP@0.5', fontsize=12)
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(axis='y', alpha=0.3)
    for xi, v in zip(x, map50_vals):
        axes[0].text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

    axes[1].bar(x, prec_vals, color=bar_colors, edgecolor='white', linewidth=1.2)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(models_list, rotation=10)
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Best Precision', fontsize=12)
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(axis='y', alpha=0.3)
    for xi, v in zip(x, prec_vals):
        axes[1].text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

    plt.suptitle('Final Model Comparison — MOT17 Pedestrian Detection',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DRIVE, 'final_comparison_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training results loaded — run training cells first')
